# Week 4 — Missing data and cultural absence

**Social and Cultural Analytics (7AAVBCS5)**

**Working in Colab?** Download this file from KEATS, then in Colab choose **File -> Upload notebook**.
The setup cell below fetches the data over the internet if there is no local `data` folder.

## What you will be able to do by the end

- Find and count missing values in a pandas DataFrame
- Report missingness as counts and percentages
- Explain why missingness can matter substantively
- Create missingness indicator columns
- Compare missingness across groups
- Join venue records to borough context
- Write a short interpretation with denominators and limitations


## Setting up


In [ ]:
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

DATA_REPO = "https://github.com/markjhill/7AAVBCS5-data.git"


def data_path(name: str) -> Path:
    """Return a usable path to a file or folder in the module's data folder."""
    here = Path.cwd()
    for candidate in (here, *here.parents):
        local = candidate / "data" / name
        if local.exists():
            return local

    downloaded = Path("7AAVBCS5-data")
    if not downloaded.is_dir():
        print("No local data folder found -- downloading the module data (once)...")
        subprocess.run(
            ["git", "clone", "--depth", "1", DATA_REPO, str(downloaded)],
            check=True,
        )
    return downloaded / name


## 1. Load the London venue data

We will use the London cultural infrastructure venue file. One row is one cultural venue.


In [ ]:
venues = pd.read_csv(data_path("london_cultural_venues_2023.csv"))
venues.head()


**Task 1.1.** Print the shape of `venues`, then inspect the columns with `.info()`.

In a text cell, answer: what does one row represent?


In [ ]:
# write your code here


## 2. Finding the gaps

A first missing-data check should ask two questions:

- How many values are missing?
- What percentage of the column is missing?


In [ ]:
# A worked example: one column.
print("missing websites:", venues["website"].isna().sum())
print("percentage missing:", round(venues["website"].isna().mean() * 100, 1))


**Task 2.1.** Build a missingness summary table with one row per column and two columns:

- `missing`
- `missing_%`

Sort it so the columns with the most missing values appear first.


In [ ]:
# write your code here


**Task 2.2.** Which three columns have the most missing values? For each one, write one sentence about why that missingness might matter for cultural analysis.


## 3. Missing values in pandas

pandas gives us methods for detecting missing values. Use `.isna()` or `.notna()`, not equality checks.


In [ ]:
example = pd.Series(["website", None, "website", np.nan])
print(example)
print(example.isna())
print(np.nan == np.nan)


**Task 3.1.** In a text cell, explain why `== np.nan` is not a safe way to find missing values.


## 4. Missingness indicators

A missingness indicator turns a gap into a variable we can analyse.


In [ ]:
venues = venues.copy()
venues["has_website"] = venues["website"].notna()
venues["missing_postcode"] = venues["postcode"].isna()

venues[["name", "venue_type", "has_website", "missing_postcode"]].head()


**Task 4.1.** Count how many venues have a website recorded, and how many do not.

Hint: use `.value_counts()` on `has_website`.


In [ ]:
# write your code here


**Task 4.2.** What percentage of venues have no postcode recorded?


In [ ]:
# write your code here


## 5. Is missingness patterned by venue type?

A total count is useful, but it does not tell us whether missingness is evenly spread across the dataset.


In [ ]:
website_by_type = (
    venues.groupby("venue_type")
    .agg(
        venues=("venue_id", "size"),
        website_share=("has_website", "mean"),
    )
)
website_by_type["website_share"] = (website_by_type["website_share"] * 100).round(1)
website_by_type.sort_values("website_share").head(10)


**Task 5.1.** Look at the ten venue types with the lowest website share. Before interpreting them, check the number of rows for each type. Which results are based on large groups, and which might be unstable because the group is small?


In [ ]:
# write your code here


**Task 5.2.** Write two sentences interpreting the website pattern by venue type. Be careful not to claim more than the data supports.


## 6. Is missingness patterned by borough?

Now compare missingness by borough. This is still venue-level data, grouped by borough name.


In [ ]:
website_by_borough = (
    venues.groupby("borough_name")
    .agg(
        venues=("venue_id", "size"),
        website_share=("has_website", "mean"),
        postcode_missing=("missing_postcode", "mean"),
    )
)
website_by_borough[["website_share", "postcode_missing"]] = (
    website_by_borough[["website_share", "postcode_missing"]] * 100
).round(1)
website_by_borough.sort_values("website_share").head(10)


**Task 6.1.** Which boroughs have the lowest website share? How many venues are those percentages based on?


In [ ]:
# write your code here


**Task 6.2.** Which boroughs have the highest postcode missingness? What might this affect analytically?


In [ ]:
# write your code here


## 7. Strategies: delete, fill, or keep

A missing-data strategy should follow from the question you are asking.


In [ ]:
print("rows before:", len(venues))
print("rows after dropna():", len(venues.dropna()))
print("rows with website:", len(venues.dropna(subset=["website"])))


**Task 7.1.** Explain the difference between `venues.dropna()` and `venues.dropna(subset=["website"])`.

Which one is more defensible for an analysis of website availability, and why?


**Task 7.2.** Create a new column called `website_status` that uses the website where one exists and the text `"No website recorded"` where it does not.

Do not overwrite the original `website` column.


In [ ]:
# write your code here


## 8. Joining borough context

The venue file has one row per venue. The borough context file has one row per borough.

A join lets us attach borough-level information to every venue in that borough.


In [ ]:
borough_context = pd.read_csv(data_path("london_cultural_infrastructure_2023_borough_context.csv"))
borough_context[["borough_name", "population_2015", "ptal_score_2014", "website_share"]].head()


**Task 8.1.** Join the borough context columns below onto `venues` using `borough_name`.

Keep all rows from `venues`.


In [ ]:
# columns to join:
context_columns = [
    "borough_name",
    "population_2015",
    "median_household_income_2012_13",
    "ptal_score_2014",
    "website_share",
]

# write your merge here


**Task 8.2.** Check the join.

Print:

- the number of rows before the join
- the number of rows after the join
- how many rows have missing `ptal_score_2014`


In [ ]:
# write your code here


**Task 8.3.** Create a column called `high_ptal` where `ptal_score_2014 >= 5`. Then use `pd.crosstab()` to compare `high_ptal` with `has_website` as row percentages.


In [ ]:
# write your code here


## 9. Interpretation

**Task 9.1.** Write a short paragraph for a data overview section. It should include:

- which fields are most missing
- one pattern by venue type or borough
- one sentence about why the missingness might matter
- one limitation or caution


## Stretch tasks

**Stretch 1.** Use `pd.crosstab()` to compare `venue_type` with `missing_postcode` as row percentages. Which venue types have the highest postcode missingness?

**Stretch 2.** Make a filtered table of boroughs with at least 50 venues and below 70% website share.

**Stretch 3.** Use the joined data to compare website availability above and below the median household income.


## Before next week

Keep the original data intact. When you create a derived column, give it a clear name and write down the decision that produced it.
